[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/youssefHosni/prompt-engineering-book-2e/blob/main/notebooks/ch01_instruction_following.ipynb)

# Chapter 1 — How Instruction-Tuned Models Work

| § | what it shows |
|---|---|
| 1.3 | "exactly three words" — precise, checkable, and not reliably followed |
| 1.5 | a product that does not exist, asked about confidently |
| 1.6 | tokens billed vs. words returned: the cheapest instrument you have |

In [1]:
# Colab only: fetch the shared helpers and install the SDKs.
# Running locally? Skip this cell -- you already have them.
import os, sys
if "google.colab" in sys.modules:
    !pip -q install anthropic openai python-dotenv
    !git clone -q https://github.com/youssefHosni/prompt-engineering-book-2e /content/peb-book
    sys.path.insert(0, "/content/peb-book")
    from getpass import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")


In [2]:
from peb import ask

RUNS = 6

## §1.3 An instruction that is precise and still not followed

Nothing here is ambiguous. The model simply does not count.

In [3]:
prompt_three_words = (
    "Summarise the following in exactly three words.\n\n"
    "The 2024 revision of the building code raised the minimum insulation "
    "requirement for new residential roofs from R-38 to R-49, "
    "effective January 2025."
)

hits = 0
for i in range(1, RUNS + 1):
    text, _ = ask(prompt_three_words, max_tokens=4000)
    n = len(text.split())
    hits += (n == 3)
    print(f"run {i}  words={n:<3} {text.strip()[:50]}")

# The count is done in code, never by asking the model. §13.4 is why.
print(f"\n{hits}/{RUNS} runs hit exactly three words.")

run 1  words=3   Roof insulation increased.


run 2  words=3   Insulation requirements increased


run 3  words=3   Roof insulation increased


run 4  words=3   Roof insulation increased.


run 5  words=3   Roof insulation increased.


run 6  words=3   Roof insulation increased

6/6 runs hit exactly three words.


## §1.5 A product that does not exist

The first edition's headline example. Worth re-running: this is one of the
2023-era failure modes that largely does **not** reproduce on current frontier
models, and the chapter says so rather than pretending otherwise.

In [4]:
text, _ = ask("Tell me about the AeroGlide UltraSlim Smart Toothbrush by Boie",
              max_tokens=4000)
print(text.strip()[:600])

hedged = any(w in text.lower() for w in
             ("don't have", "do not have", "not aware", "no record", "unable to find"))
print(f"\nhedged rather than invented: {hedged}")

I don't have any information about a product by that name, and I'd be cautious about it — "AeroGlide UltraSlim Smart Toothbrush" doesn't match anything I'm aware of.

Boie is a real company, though. Boie USA is a US-based personal care brand known for making toothbrushes and body scrubbers out of a soft, antimicrobial thermoplastic elastomer, with a focus on durability, recyclability, and replaceable heads rather than fully disposable products. Their toothbrushes are generally simple manual or basic electric designs, not "smart" connected devices as far as I know.

So a few possibilities:

- T

hedged rather than invented: True


## §1.6 Tokens billed vs. words returned

The gap between what you pay for and what you get back is the cheapest
diagnostic available, and you will only look at it if it is already in your
hand — which is why `ask()` returns usage alongside the text.

In [5]:
prompt_api = ("How do I use the batch_stream() method on the Anthropic Python "
              "client? Show a short code example.")

for label, p in (("three words", prompt_three_words), ("api question", prompt_api)):
    text, usage = ask(p, max_tokens=4000)
    words = len(text.split())
    print(f"{label:<14} in={usage.input_tokens:<5} out={usage.output_tokens:<5} "
          f"words={words:<5} ratio={usage.output_tokens / max(words, 1):.2f}")

print("\nA ratio far above ~1.4 means tokens went somewhere the text did not.")

three words    in=67    out=203   words=3     ratio=67.67


api question   in=36    out=2043  words=290   ratio=7.04

A ratio far above ~1.4 means tokens went somewhere the text did not.
